# Week 30, Day 3: Overriding and the MRO

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w030d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Report:
    def render(self):
        return self.header() + " | " + self.body()

    def header(self):
        return "REPORT"

    def body(self):
        return "(nothing yet)"

class SalesReport(Report):
    def body(self):
        return "12 sold"

print(Report().render())
print(SalesReport().render())

In [ ]:
class Shape:
    def area(self):
        return 0

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self, pi):                         # a new required argument
        return pi * self.radius ** 2

shapes = [Square(2), Circle(1)]
print(sum(s.area() for s in shapes))

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"{type(self).__name__}({self.x}, {self.y})"

class LabelledPoint(Point):
    def __init__(self, x, y, label):
        super().__init__(x, y)
        self.label = label

    def __repr__(self):
        return f"{super().__repr__()[:-1]}, {self.label!r})"

print(Point(1, 2), LabelledPoint(3, 4, "Kano"))

In [ ]:
class Receipt:
    def render(self, items):
        lines = [self.title()]
        for name, pence in items:
            lines.append(self.line(name, pence))
        lines.append(self.footer(sum(p for _, p in items)))
        return "\n".join(lines)

    def title(self):
        return "RECEIPT"

    def line(self, name, pence):
        return f"{name}: {pence / 100:.2f}"

    def footer(self, total):
        return f"Total: {total / 100:.2f}"

class ShortReceipt(Receipt):
    def title(self):
        return "--"

    def line(self, name, pence):
        return name[:8]

items = [("Plantain chips", 150), ("Ginger beer", 120)]
print(Receipt().render(items))
print(ShortReceipt().render(items))

In [ ]:
class Swimmer:
    def move(self):
        return "swims"

    def breathe(self):
        return "with gills"

class Walker:
    def move(self):
        return "walks"

    def legs(self):
        return 4

class Frog(Walker, Swimmer):
    pass

f = Frog()
print(f.move(), f.breathe(), f.legs())
print([cls.__name__ for cls in Frog.__mro__])

In [ ]:
class DictMixin:
    """Adds to_dict() to any class whose attributes are simple values."""

    def to_dict(self):
        return {"kind": type(self).__name__, **vars(self)}

class Book:
    def __init__(self, title, pages):
        self.title, self.pages = title, pages

class SavedBook(DictMixin, Book):
    pass

class Train(DictMixin):
    def __init__(self, code, to):
        self.code, self.to = code, to

print(SavedBook("Aké", 230).to_dict())
print(Train("1A23", "Edinburgh").to_dict())

**Predict before you run:** what will this print?



In [ ]:
class Base:
    def __init__(self):
        print("Base")

class Left(Base):
    def __init__(self):
        print("Left")
        super().__init__()

class Right(Base):
    def __init__(self):
        print("Right")
        super().__init__()

class Both(Left, Right):
    def __init__(self):
        print("Both")
        super().__init__()

Both()
print([cls.__name__ for cls in Both.__mro__])

In [ ]:
class Account:
    def __init__(self, balance=0):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount
        return self.balance

class AuditMixin:
    def withdraw(self, amount):
        left = super().withdraw(amount)             # the next class in the MRO: Account
        print(f"audit: took {amount}, {left} left")
        return left

class AuditedAccount(AuditMixin, Account):
    pass

a = AuditedAccount(500)
a.withdraw(120)
print([cls.__name__ for cls in AuditedAccount.__mro__])

In [ ]:
class Item:
    pass

class Book(Item):
    pass

class Odd(Item, Book):      # Item first, but Book must come before its parent Item
    pass

**Predict before you run:** what will this print?



In [ ]:
class Counter:
    def __init__(self):
        self._count = 0

    def tick(self):
        self._count += 1
        return self._count

class PageCounter(Counter):
    def __init__(self, first_page):
        super().__init__()
        self._count = first_page          # meant for the child's own use

p = PageCounter(100)
print(p.tick(), p.tick())

In [ ]:
class Counter:
    def __init__(self):
        self.__count = 0

    def tick(self):
        self.__count += 1
        return self.__count

class PageCounter(Counter):
    def __init__(self, first_page):
        super().__init__()
        self.__count = first_page

    def page(self):
        return self.__count

p = PageCounter(100)
print(p.tick(), p.tick(), p.page())
print(sorted(vars(p)))

> **Key idea.** A good override keeps the parent's arguments and kind of result, so any code that works with the parent works with the child. A parent's method that calls `self.hook()` runs the child's version, so a parent can lay out a job and let children fill in steps. With several parents, Python searches the MRO (`C.__mro__`): each class before its parents, parents in the listed order, each class once. `super()` means "the next class in the MRO", which runs every `__init__` once in a diamond and lets mixins (listed first) extend the main class. Double-underscore names are mangled with the class's name, so a parent's and a child's internals can't clash.

### Exercise: Notices

`Notice` renders as `[i] text`. Write `Caution`, a `Notice` whose icon is `!`, and `Closure`, a `Notice` whose icon is `X` and whose whole rendered line is in capitals: `[X] LIBRARY SHUT ON MONDAY`. Override `icon` in both; `Closure` also extends `render` with `super()`. Don't change `Notice`.

In [ ]:
class Notice:
    def __init__(self, text):
        self.text = text

    def icon(self):
        return "i"

    def render(self):
        return f"[{self.icon()}] {self.text}"

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_render():\n    \"rendering\"\n    N, Ca, Cl = ns[\"Notice\"], ns[\"Caution\"], ns[\"Closure\"]\n    assert N(\"Open at 9\").render() == \"[i] Open at 9\"\n    assert Ca(\"Wet floor\").render() == \"[!] Wet floor\"\n    assert Cl(\"Library shut on Monday\").render() == \"[X] LIBRARY SHUT ON MONDAY\"\n    assert issubclass(Ca, N) and issubclass(Cl, N) and \"render\" not in vars(Ca)\n    assert Cl(\"a\").icon() == \"X\" and Ca(\"a\").icon() == \"!\"\n    assert \"super()\" in source, \"Closure.render should use super()\"")

### Exercise: Exporters

`Exporter.export(columns, rows)` lays out a table: the header lines from `header(columns)`, then one line per row from `line(row)`. Write two children that fill in the hooks:

- `CsvExporter`: the header is one line of column names joined by commas; each row is its values joined by commas.
- `MarkdownExporter`: the header is two lines, `| name | price |` and `|---|---|` (one `---|` per column); each row is like `| Bread | 135 |`.

Don't override `export`.

In [ ]:
class Exporter:
    def export(self, columns, rows):
        lines = self.header(columns) + [self.line(row) for row in rows]
        return "\n".join(lines)

    def header(self, columns):
        return []

    def line(self, row):
        raise NotImplementedError("children must define line()")

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_csv():\n    \"CSV\"\n    C = ns[\"CsvExporter\"]\n    assert C().export([\"name\", \"price\"], [(\"Bread\", 135), (\"Milk\", 95)]) == \"name,price\\nBread,135\\nMilk,95\"\n    assert C().export([\"a\"], []) == \"a\"\ndef test_markdown():\n    \"Markdown\"\n    M = ns[\"MarkdownExporter\"]\n    got = M().export([\"name\", \"price\"], [(\"Bread\", 135)])\n    assert got == \"| name | price |\\n|---|---|\\n| Bread | 135 |\", f\"got {got!r}\"\n    assert M().export([\"x\", \"y\", \"z\"], [(1, 2, 3)]).split(\"\\n\")[1] == \"|---|---|---|\"\ndef test_family():\n    \"hooks only\"\n    E, C, M = ns[\"Exporter\"], ns[\"CsvExporter\"], ns[\"MarkdownExporter\"]\n    assert issubclass(C, E) and issubclass(M, E)\n    assert \"export\" not in vars(C) and \"export\" not in vars(M), \"inherit export\"\n    try:\n        E().export([\"a\"], [(1,)])\n        assert False, \"Exporter itself should still raise NotImplementedError\"\n    except NotImplementedError:\n        pass")

### Exercise: Two mixins

Write two mixins that work with any class whose attributes are simple values:

- `ReprMixin` gives `__repr__` in the form `Train(code='1A23', to='York')`: the class name, then each attribute in the order it was set, as `name=repr(value)`, separated by `, `.
- `EqMixin` gives `__eq__`: equal when the other object has **exactly** the same class and the same attributes; `NotImplemented` otherwise.

Then write `Train(code, to)` using both mixins.

In [ ]:
class ReprMixin:
    ___

class EqMixin:
    ___

class Train(___):
    def __init__(self, code, to):
        self.code = code
        self.to = to

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_train():\n    \"trains\"\n    T = ns[\"Train\"]\n    assert repr(T(\"1A23\", \"York\")) == \"Train(code='1A23', to='York')\", f\"got {T('1A23', 'York')!r}\"\n    assert T(\"1A23\", \"York\") == T(\"1A23\", \"York\") and T(\"1A23\", \"York\") != T(\"1A23\", \"Leeds\")\n    assert T(\"1A23\", \"York\").__eq__(\"1A23\") is NotImplemented\ndef test_other():\n    \"mixins on another class\"\n    R, E = ns[\"ReprMixin\"], ns[\"EqMixin\"]\n    class Stop(R, E):\n        def __init__(self, name, mins):\n            self.name = name\n            self.mins = mins\n    class Other(R, E):\n        def __init__(self, name, mins):\n            self.name = name\n            self.mins = mins\n    assert repr(Stop(\"Kano\", 5)) == \"Stop(name='Kano', mins=5)\"\n    assert Stop(\"a\", 1) == Stop(\"a\", 1) and Stop(\"a\", 1) != Other(\"a\", 1), \"different classes aren't equal\"\n    assert repr(R()) == \"ReprMixin()\"")

### Exercise: A logging mixin

Write `LogMixin`, which records successful withdrawals and deposits in a list `self.log` as strings like `"withdraw 30"` and `"deposit 50"`. Its methods extend the next class in the MRO with `super()`, and only record **after** the call succeeds. Give it an `__init__` that passes every argument on (`def __init__(self, *args, **kwargs)` and `super().__init__(*args, **kwargs)`) and then makes the empty log. Then write `LoggedAccount`, using `LogMixin` and `Account`, with an empty body.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("not enough money")
        self.balance -= amount
        return self.balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_log():\n    \"logging\"\n    L = ns[\"LoggedAccount\"]\n    a = L(\"Tunde\", balance=100)\n    assert a.withdraw(30) == 70 and a.deposit(50) == 120\n    try:\n        a.withdraw(500)\n        assert False, \"the account should still refuse\"\n    except ValueError:\n        pass\n    assert a.log == [\"withdraw 30\", \"deposit 50\"], f\"log is {a.log}\"\n    assert a.owner == \"Tunde\" and L(\"x\").log == [] and L(\"x\").balance == 0\ndef test_mro():\n    \"the class order\"\n    names = [c.__name__ for c in ns[\"LoggedAccount\"].__mro__]\n    assert names == [\"LoggedAccount\", \"LogMixin\", \"Account\", \"object\"], f\"MRO is {names}\"\n    assert \"withdraw\" not in vars(ns[\"LoggedAccount\"])")

### Bug hunt: The mixin that never shouts

`LoudGreeter` should print `HELLO, AMARA!` but prints `Hello, Amara`. The classes are right; fix the order of the parents.

In [ ]:
class Greeter:
    def greet(self, name):
        return f"Hello, {name}"

class ShoutMixin:
    def greet(self, name):
        return super().greet(name).upper() + "!"

class LoudGreeter(Greeter, ShoutMixin):
    pass

print(LoudGreeter().greet("Amara"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shouting\"\n    expect_output(\"HELLO, AMARA!\")")

### Bug hunt: Registered twice

Each new `Kit` should be registered once, so this should print `1`, but it prints `2`. Fix it without changing `Device`.

In [ ]:
class Device:
    registered = 0

    def __init__(self):
        Device.registered += 1

class Phone(Device):
    def __init__(self):
        Device.__init__(self)
        self.can_call = True

class Camera(Device):
    def __init__(self):
        Device.__init__(self)
        self.can_film = True

class Kit(Phone, Camera):
    def __init__(self):
        Phone.__init__(self)
        Camera.__init__(self)

k = Kit()
print(Device.registered)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"registered once\"\n    expect_output(\"1\")\n    k = ns[\"Kit\"]()\n    assert k.can_call and k.can_film, \"a Kit can still call and film\"")

### Bug hunt: The lost total

A `Timer` adds up seconds in `_total`. `LapTimer` adds a lap count, but the total comes out wrong: it should print `90 2`, not `2 2`. Fix the clash so both values survive. Make the fix inside `Timer` with name mangling, so that whoever writes the next subclass can't make the same mistake.

In [ ]:
class Timer:
    def __init__(self):
        self._total = 0

    def add(self, seconds):
        self._total += seconds

    def total(self):
        return self._total

class LapTimer(Timer):
    def __init__(self):
        super().__init__()
        self.laps = 0

    def lap(self, seconds):
        self.add(seconds)
        self.laps += 1
        self._total = self.laps

t = LapTimer()
t.lap(40)
t.lap(50)
print(t.total(), t.laps)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both values\"\n    expect_output(\"90 2\")\n    t = ns[\"Timer\"]()\n    t.add(5)\n    assert \"_Timer__total\" in vars(t), \"Timer should keep its total in a name-mangled attribute\"")

### Puzzle: A timestamp mixin

Arrange the lines so that `StampedNote` uses a mixin that extends `Note.text` by adding the day. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B" + super().who()

class C(A):
    def who(self):
        return "C" + super().who()

class D(B, C):
    pass

class E(C, B):
    def who(self):
        return "E"

p = ___   # D().who()
q = ___   # [k.__name__ for k in D.__mro__]
r = ___   # E().who()
s = ___   # [k.__name__ for k in E.__mro__][1]
t = ___   # issubclass(D, C)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"BCA\", \"p\"\n    assert q == [\"D\", \"B\", \"C\", \"A\", \"object\"], \"q\"\n    assert (r, s, t) == (\"E\", \"C\", True)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: delivery options

A shop offers deliveries with optional extras. Build them with a base class, children for the services, and mixins for the extras.

- `Delivery(grams)`: `grams` must be an `int` above 0, otherwise `ValueError`. Class attributes `BASE = 300` and `PER_KG = 100`. `kg()` is the weight in whole kilograms, rounded **up** (1 to 1000 grams is 1, 1001 is 2). `cost()` is `BASE + PER_KG * kg()`. `extras()` returns `[]`. `describe()` returns for example `"NextDay 1kg: 850p"`, or with extras `"SecureNextDay 3kg: 1550p (insured, signed for)"`: the class name, the kilograms, the cost, and the extras in brackets joined by `, ` if there are any.
- `Standard(Delivery)` keeps the base prices. `NextDay(Delivery)` has `BASE = 700` and `PER_KG = 150`.
- `InsuredMixin` adds 250 to the cost and `"insured"` to the extras; `SignedMixin` adds 150 and `"signed for"`. Each extends `cost()` and `extras()` with `super()`, adding its own extra to the **end** of the list.
- `InsuredStandard` (insured standard delivery) and `SecureNextDay` (next day, insured and signed for, with extras in the order `insured, signed for`). Their bodies are empty: the order of the parents does the work.

`describe` is written once, in `Delivery`.

In [ ]:
import math

class Delivery:
    BASE = 300
    PER_KG = 100
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_base():\n    \"the base and services\"\n    D, S, N = ns[\"Delivery\"], ns[\"Standard\"], ns[\"NextDay\"]\n    assert [D(g).kg() for g in [1, 1000, 1001, 2500]] == [1, 1, 2, 3]\n    assert S(800).cost() == 400 and N(1000).cost() == 850 and N(2001).cost() == 1150\n    assert N(1000).describe() == \"NextDay 1kg: 850p\" and S(5).extras() == []\n    for bad in [0, -10, 2.5, \"100\"]:\n        try:\n            D(bad)\n            assert False, f\"Delivery({bad!r}) should raise ValueError\"\n        except ValueError:\n            pass\ndef test_extras():\n    \"extras\"\n    IS, SN = ns[\"InsuredStandard\"], ns[\"SecureNextDay\"]\n    assert IS(1500).cost() == 750 and IS(1500).extras() == [\"insured\"]\n    assert IS(1500).describe() == \"InsuredStandard 2kg: 750p (insured)\"\n    s = SN(2500)\n    assert s.cost() == 1550 and s.extras() == [\"insured\", \"signed for\"], f\"extras are {s.extras()}\"\n    assert s.describe() == \"SecureNextDay 3kg: 1550p (insured, signed for)\", f\"got {s.describe()!r}\"\ndef test_design():\n    \"the design\"\n    D, S, N, IM, SM, IS, SN = (ns[n] for n in [\"Delivery\", \"Standard\", \"NextDay\", \"InsuredMixin\", \"SignedMixin\", \"InsuredStandard\", \"SecureNextDay\"])\n    assert issubclass(IS, S) and issubclass(IS, IM) and issubclass(SN, N) and issubclass(SN, IM) and issubclass(SN, SM)\n    assert not issubclass(IM, D) and not issubclass(SM, D), \"mixins shouldn't inherit from Delivery\"\n    for cls in (S, N, IS, SN):\n        assert \"describe\" not in vars(cls) and \"cost\" not in vars(cls), f\"{cls.__name__} should inherit describe and cost\"\n    for cls in (IS, SN):\n        assert all(k.startswith(\"__\") for k in vars(cls)), f\"{cls.__name__}'s body should be empty\"")

## Warm-up

`render` is written once, in `Report`, and never overridden. But when it runs for a `SalesReport`, its call `self.body()` starts searching at the sales report, finds the child's `body`, and uses it. The parent's code calls the child's method without knowing the child exists. A lot of good class design rests on that.

## Good overrides

An override replaces the parent's method for the child's objects. Python only checks the **name**; it's up to you to keep the new method usable in the same way. The test is simple: **any code that works with the parent should still work when given a child**.

That means an override should:

- accept the same arguments (it may add new ones, if they have defaults);
- return the same kind of thing (a number if the parent returns a number, a string if it returns a string);
- not raise errors that the parent never raised for the same input.

Here's an override that breaks the rule:

The loop worked for every `Shape` until a circle arrived. Giving `pi` a default (`def area(self, pi=3.14159)`) or using `math.pi` inside would keep the override safe.

Special methods are overridden the same way. A child that adds an attribute which matters for equality should override `__eq__` (and `__hash__` with it), and a child with more to show can override `__repr__`:

## Parents that call their children

The warm-up showed a design you'll meet everywhere: the parent writes the **outline** of a job in one method, and calls smaller methods (often called **hooks**) for the parts that differ. Each child overrides only the hooks it needs.

- The order of the steps, and the adding up, live in one place. A fix to `render` fixes every kind of receipt.
- `ShortReceipt` overrides two hooks and keeps the third.
- When there's no sensible default for a hook, the parent raises `NotImplementedError` (as `Shape.area` did on Day 1) so a child that forgets it fails loudly. Week 32's abstract base classes make this stricter.

## More than one parent

A class can list several parents: `class C(A, B):`. It inherits from all of them, and when two parents both have a name, the one listed **first** wins:

The common, safe use of this is the **mixin**: a small class that adds one ability (a repr, a way to save, logging) and isn't meant to be used on its own. Mixins usually have no `__init__` and rely only on `self`:

Mixins go **first** in the list, before the main parent, so that their methods win if both define the same name. Keep multiple inheritance to mixins like this: deep families with several full parents are hard to follow, and Day 5 shows a simpler alternative.

## The MRO and super()

Every class has one search order, its **method resolution order** (MRO), in `C.__mro__` (or `C.mro()`). Python builds it so that:

1. a class comes before its parents;
2. parents keep the order they're listed in;
3. each class appears once, even if it's reachable two ways.

The third rule matters in a **diamond**, where two parents share a grandparent:

Look at `Left`: its `super().__init__()` ran **`Right`**'s `__init__`, though `Right` isn't `Left`'s parent. That's because `super()` doesn't mean "my parent". It means **the next class in the MRO of the object being built**. For a `Both` object that order is `Both, Left, Right, Base, object`, so after `Left` comes `Right`. Each `__init__` runs exactly once, and `Base` runs last.

That's also why calling a parent by name (`Base.__init__(self)`) causes trouble with several parents: in a diamond, `Left` and `Right` would each call `Base.__init__` directly, and it would run twice. With `super()` everywhere, every class gets one turn.

Because `super()` follows the MRO, a mixin can extend a method of a class it doesn't even inherit from:

`AuditMixin`'s only parent is `object`, which has no `withdraw`. But in an `AuditedAccount`, the class after `AuditMixin` is `Account`, so that's where `super().withdraw` goes.

Not every order can be built. If the rules contradict each other, Python refuses to create the class:

## Name mangling, at last

Week 29, Day 5 said that double-underscore names exist to stop a child and its parent clashing. Here's the clash. Both classes below use `_count` for their own private purposes:

The child's `_count` and the parent's are the same attribute on the same object, so the child's starting page wiped out the parent's tally. If the parent had written `self.__count`, Python would have stored it as `_Counter__count`, and a child writing `self.__count` would get `_PageCounter__count`: two separate attributes.

That's the whole purpose of mangling. Most classes are fine with a single underscore; use two for internal attributes of a class that other people will subclass without reading its code.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick any class family from this week and print its `__mro__`; then predict, before running, which class's method runs for each call. Tomorrow: polymorphism and duck typing, where one loop handles many kinds of object, and Python cares what an object can do rather than what it inherits from.